## load required pacakages

In [1]:
# main.py
import argparse
import os
import re
import time
import numpy as np
import pandas as pd
import sklearn
import json
import csv
from Bio.Seq import Seq
from Bio import SeqIO
from evcouplings.compare import DistanceMap
from Bio.Data import CodonTable
    # Split data into training and testing sets
from sklearn.model_selection import train_test_split
from joblib import Parallel, delayed
import joblib

In [2]:
from scipy.optimize import check_grad
from scipy.sparse import issparse
from sklearn.metrics import mean_squared_error,precision_recall_curve, auc,roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, Lasso, RidgeCV,RidgeClassifierCV,Ridge,LogisticRegressionCV


In [126]:
from data_loading import *
from model_training import train_ridge_model
from data_preprocessing import *
from sequence_translation import *
from distance_processing import *
from evaluation import *

## initialize by gene name

In [4]:
results= []

In [5]:
gene_name='inhA'
genes_of_interest=gene_name.split(',')

In [6]:
# Set random seed for reproducibility
seed_everything(42)

## required preprocessing data loading

In [13]:
# Load phenotype data and other initializations
phenotype_paths = ["../data/master_table_resistance.csv","../data/cryptic_dataset.csv"]

In [17]:
gene_details = pd.read_csv("../data/all_drug_genes_locations.csv")
# Filter the DataFrame for the genes of interest
filtered_df = gene_details [gene_details ['gene_name'].str.contains('|'.join(genes_of_interest), case=False, na=False)]
drug_name = filtered_df['drug_full'].values[0].upper()

uniprot = filtered_df['Uniprot'].values[0]
entry = filtered_df['Entry'].values[0]
print(f"The drug associated with the gene {gene_name} is: {drug_name}")

The drug associated with the gene inhA is: ISONIAZID


In [18]:
# Load data
phenotype_data = load_phenotype_data(phenotype_paths,drug_name).reset_index()
phenotype_data

Number of records in phenotype data: 29576


/work/pi_annagreen_umass_edu/mahbuba/Data-Curation-for-MTB/protein-tasks/protein_translation/data_loading.py:11: DtypeWarning: Columns (3,4,7,8,9,12,17,20,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


,level_0,index,Isolate,AMIKACIN,AMOXICILLIN,AMOXICILLIN_CLAVULANATE,CAPREOMYCIN,CIPROFLOXACIN,CLARITHROMYCIN,CLOFAZIMINE,...,biosample,internal,path,Isolate_original,accessions,Unnamed: 0,UNIQUEID,ID,ROLLINGDB_ID,BIOSAMPLE_ACCESSION
0,0,0.0,00R0025,R,NaN,NaN,S,NaN,NaN,NaN,...,NaN,NaN,/n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...,00R0025,SAMN05916422,NaN,NaN,NaN,NaN,NaN
1,1,1.0,00R0086,R,NaN,NaN,R,NaN,S,NaN,...,NaN,NaN,/n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...,00R0086,SAMN05918543,NaN,NaN,NaN,NaN,NaN
2,2,2.0,00R0178,R,NaN,NaN,R,NaN,R,NaN,...,NaN,NaN,/n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...,00R0178,SAMN05918545,NaN,NaN,NaN,NaN,NaN
3,3,3.0,00R0223,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,/n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...,00R0223,ERS2401270,NaN,NaN,NaN,NaN,NaN
4,4,4.0,00R0312,NaN,NaN,NaN,R,NaN,NaN,NaN,...,NaN,NaN,/n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...,00R0312,ERS2401327,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29571,29571,NaN,site.14.subj.4445.lab.4445.iso.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,/n/data1/hms/dbmi/farhat/rollingDB/cryptic_out...,NaN,NaN,10424.0,site.14.subj.4445.lab.4445.iso.1,site.14.subj.4445.lab.4445.iso.1,ERR4813816,ERS5303976
29572,29572,NaN,site.14.subj.4446.lab.4446.iso.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,/n/data1/hms/dbmi/farhat/rollingDB/cryptic_out...,NaN,NaN,10425.0,site.14.subj.4446.lab.4446.iso.1,site.14.subj.4446.lab.4446.iso.1,ERR4813817,ERS5303977
29573,29573,NaN,site.14.subj.448.lab.448.iso.1,S,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,/n/data1/hms/dbmi/farhat/rollingDB/cryptic_out...,NaN,NaN,10426.0,site.14.subj.448.lab.448.iso.1,site.14.subj.448.lab.448.iso.1,ERR4813863,ERS5304023
29574,29574,NaN,site.14.subj.687.lab.687.iso.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,/n/data1/hms/dbmi/farhat/rollingDB/cryptic_out...,NaN,NaN,10427.0,site.14.subj.687.lab.687.iso.1,site.14.subj.687.lab.687.iso.1,ERR4813864,ERS5304024


In [19]:
phenotype_data=phenotype_data.drop(['level_0'], axis=1)

In [20]:
phenotype_data["Isolate_mapped"] = [path.split("/")[-1].split(".vcf")[0].split(".cut")[0] for path in phenotype_data.path]
phenotype_mapping = dict(zip(phenotype_data['Isolate_mapped'], phenotype_data[drug_name.upper()]))

In [21]:
# phenotype_mapping

In [22]:
fasta_dir = "../data/cryptic_fasta_files"

In [23]:
subset_alignment=''
h37rv_nongap_indices=[]
for index, row in filtered_df.iterrows():
    # Extract data from the current row
    fasta_filename = row['filename']
    print(fasta_filename)
    print(f"Processing file: {fasta_filename}")
    start_index = row['start_position_on_the_genomic_accession']
    end_index = row['end_position_on_the_genomic_accession']
    uniprot = row['Uniprot']
    entry = row['Entry']
    drug = row['drug_full']
    drug_code = row['Drug']
    gene = row['gene_name']
    orientation = row['orientation']
    file_path = os.path.join(fasta_dir, fasta_filename)
    filename = os.path.basename(file_path)


    # Load fasta files
    alignment = load_alignment(file_path)
    # print(f"Loaded alignment matrix for {filename}")
    print(f"Gene: {gene}, Drug: {drug}")
    print(f"gene shape: {alignment.matrix.shape}")

    # Load h37rv reference sequence
    h37rv_numbers = np.load("../data/X_matrix_H37RV_coords.npy")
    h37rv_ref = alignment.select(sequences=[alignment.id_to_index["MT_H37Rv"]])
    # Find the closest index to the gene start and end
    subset_alignment, column_index, start_index, end_index = sort_gene_indices(h37rv_numbers, start_index, end_index, alignment)
    print("col index", column_index)
    if column_index is not None:
        print(f"Using column {column_index} for alignment selection.")
        print(f"Reference numbers for this column: {h37rv_numbers[:, column_index]}")
    else:
        raise ValueError(f"No valid column found for gene start {start_index}.")

    h37rv_alignment = select_subset_alignment(h37rv_ref, start_index, end_index,h37rv_numbers[:, column_index])
    h37rv_nongap_indices = np.where(h37rv_alignment[0] != "-")[0]
    h37rv_sequence_str = ''.join(h37rv_alignment[0][h37rv_nongap_indices])

FabG1-inhA.fasta
Processing file: FabG1-inhA.fasta
Gene: inhA, Drug: isoniazid
gene shape: (31452, 2594)
Processed subset alignment for gene start 1674201.0 and end 1675010.0 in column 13
col index 13
Using column 13 for alignment selection.
Reference numbers for this column: [1672457. 1672458. 1672459. ...       0.       0.       0.]


In [24]:
df_h37rv = pd.DataFrame(h37rv_numbers)
df_h37rv

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,2517695.0,4407528.0,1833378.0,4036731.0,4239663.0,4266953.0,1471576.0,4326004.0,2725477.0,1917755.0,2153235.0,781311.0,759609.0,1672457.0,2713783.0,4997.0,4043041.0,2287883.0
1,2517696.0,4407529.0,1833379.0,4036732.0,4239664.0,4266954.0,1471577.0,4326005.0,2725478.0,1917756.0,2153236.0,781312.0,759610.0,1672458.0,2713784.0,4998.0,4043042.0,2287884.0
2,2517697.0,4407530.0,1833380.0,4036733.0,4239665.0,4266955.0,1471578.0,4326006.0,2725479.0,1917757.0,2153237.0,781313.0,759611.0,1672459.0,2713785.0,4999.0,4043043.0,2287885.0
3,2517698.0,4407531.0,1833381.0,4036734.0,4239666.0,4266956.0,1471579.0,4326007.0,2725480.0,1917758.0,2153238.0,781314.0,759612.0,1672460.0,2713786.0,5000.0,4043044.0,2287886.0
4,2517699.0,4407532.0,1833382.0,4036735.0,4239667.0,4266957.0,1471580.0,0.0,2725481.0,1917759.0,2153239.0,781315.0,759613.0,1672461.0,2713787.0,5001.0,4043045.0,2287887.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10286,0.0,0.0,0.0,0.0,4249805.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10287,0.0,0.0,0.0,0.0,4249806.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10288,0.0,0.0,0.0,0.0,4249807.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10289,0.0,0.0,0.0,0.0,4249808.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [26]:
ref_sequence='atgacaggactgctggacggcaaacggattctggttagcggaatcatcaccgactcgtcgatcgcgtttcacatcgcacgggtagcccaggagcagggcgcccagctggtgctcaccgggttcgaccggctgcggctgattcagcgcatcaccgaccggctgccggcaaaggccccgctgctcgaactcgacgtgcaaaacgaggagcacctggccagcttggccggccgggtgaccgaggcgatcggggcgggcaacaagctcgacggggtggtgcattcgattgggttcatgccgcagaccgggatgggcatcaacccgttcttcgacgcgccctacgcggatgtgtccaagggcatccacatctcggcgtattcgtatgcttcgatggccaaggcgctgctgccgatcatgaaccccggaggttccatcgtcggcatggacttcgacccgagccgggcgatgccggcctacaactggatgacggtcgccaagagcgcgttggagtcggtcaacaggttcgtggcgcgcgaggccggcaagtacggtgtgcgttcgaatctcgttgccgcaggccctatccggacgctggcgatgagtgcgatcgtcggcggtgcgctcggcgaggaggccggcgcccagatccagctgctcgaggagggctgggatcagcgcgctccgatcggctggaacatgaaggatgcgacgccggtcgccaagacggtgtgcgcgctgctgtctgactggctgccggcgaccacgggtgacatcatctacgccgacggcggcgcgcacacccaattgctctag'

In [27]:
print(f"Reference Sequence Length: {len(ref_sequence)}")
print(f"Extracted Sequence Length: {len(h37rv_sequence_str)}")


Reference Sequence Length: 810
Extracted Sequence Length: 809


In [28]:
alignment.matrix.shape

(31452, 2594)

In [29]:
filenames = [path.split("/")[-1].split(".cut")[0] for path in subset_alignment.ids]
len(filenames)

31452

In [30]:
filename_to_sequence = {}
for i, filename in enumerate(filenames):
    if filename not in filename_to_sequence and i < len(subset_alignment):
        filename_to_sequence[filename] = ''.join(subset_alignment[i])
filenames = list(filename_to_sequence.keys()) 
len(filenames)

31312

In [31]:

## create frequency dataframe from who catalog
who_catalog_path = '/work/pi_annagreen_umass_edu/mahbuba/phenotype_prediction/data/WHO-UCN-TB-2023.7-eng.xlsx' 
who_catalog = pd.read_excel(who_catalog_path, sheet_name='Catalogue_master_file',header=2)
# Extract relevant columns
frequency_df = who_catalog[['drug','gene', 'mutation','variant','effect','Present_R','Present_S','Present_SOLO_R','FINAL CONFIDENCE GRADING']]
print(frequency_df.head())

# subset for target genes
frequency_df = frequency_df[frequency_df['variant'].str.startswith(tuple(genes_of_interest))]

# Display the filtered DataFrame
print(len(frequency_df))
# Discard rows that have 'ins' or 'del' in the variant column
frequency_df = frequency_df[~frequency_df['variant'].str.contains('ins|del')]
frequency_df['variant_position'] =frequency_df['variant'].apply(
    lambda x: int(re.search(r'(\d+)', x).group(1)) if isinstance(x, str) and re.search(r'(\d+)', x) else None
)

frequency_threshold = 5  # Define your threshold here
# rare_variants, common_variants = classify_variants_by_frequency(frequency_df, frequency_threshold)
## load the mutations file
mutation_file_path = "/work/pi_annagreen_umass_edu/mahbuba/phenotype_prediction/data/mutations_with_one_letter_all.csv"
variants_df = load_variants(mutation_file_path)

# List all FASTA files in the directory
fasta_files = [f for f in os.listdir(fasta_dir) if f.endswith('.fasta')]

distmap_path = f"/work/pi_annagreen_umass_edu/mahbuba/phenotype_prediction/data/structure_plus_mutation/positive_control_data/distmaps/{uniprot}/{entry}"
dist_map = DistanceMap.from_file(distmap_path)


/work/pi_annagreen_umass_edu/mahbuba/protein/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


       drug  gene   mutation         variant              effect  Present_R  \
0  Amikacin  bacA   c.102G>A   bacA_c.102G>A  synonymous_variant        0.0   
1  Amikacin  bacA  c.1044G>A  bacA_c.1044G>A  synonymous_variant       91.0   
2  Amikacin  bacA   c.105C>G   bacA_c.105C>G  synonymous_variant        0.0   
3  Amikacin  bacA  c.1065T>G  bacA_c.1065T>G  synonymous_variant        0.0   
4  Amikacin  bacA  c.1080G>A  bacA_c.1080G>A  synonymous_variant        0.0   

   Present_S  Present_SOLO_R    FINAL CONFIDENCE GRADING  
0        1.0             NaN  4) Not assoc w R - Interim  
1     1112.0             NaN            5) Not assoc w R  
2        1.0             NaN  4) Not assoc w R - Interim  
3        3.0             NaN  4) Not assoc w R - Interim  
4        2.0             NaN  4) Not assoc w R - Interim  
520


# Data preparation: from DNA seq to protein translation to feature matrix generation

## save gene specific sequence data csv with filename, nucleotide sequence, associated phenotype and seq len

In [34]:
# Define the output file
output_file = "../data/sequence_data_csv/"+gene_name+"_sequence_data.csv"


# Prepare the data list
data_list = []
for i in range(len(filenames)):
    filename = filenames[i]

    if filename in phenotype_mapping and filename in filename_to_sequence:

        # Fetch the corresponding phenotype
        phenotype = phenotype_mapping[filename]
        # Fetch the corresponding sequence

        sequence = filename_to_sequence[filename]
        # print("len of sequence before non gap indices", len(sequence))
        # print(np.array(list(sequence)))

        ## we include the nongap indices here so that we're only looking at mutations relative to h37rv
        sequence = "".join(np.array(list(sequence))[h37rv_nongap_indices])
        
        # Append to list
        data_list.append([filename, sequence, phenotype, len(sequence)])
        # print("len of sequence after non gap indices", len(sequence))

# Save data to CSV
with open(output_file, mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["Filename", "Sequence", "Phenotype","seq_len"])  # Header
    writer.writerows(data_list)

print(f"CSV file '{output_file}' saved successfully!")

CSV file '../data/sequence_data_csv/inhA_sequence_data.csv' saved successfully!


### convert nucleotide sequence to protein sequence

In [35]:
file_path="../data/sequence_data_csv/"+gene_name+"_sequence_data.csv"
gene_sequence_data=pd.read_csv(file_path)

In [36]:
gene_sequence_data['Sequence'][0]

'atgacaggactgctggacggcaaacggattctggttagcggaatcatcaccgactcgtcgatcgcgtttcacatcgcacgggtagcccaggagcagggcgcccagctggtgctcaccgggttcgaccggctgcggctgattcagcgcatcaccgaccggctgccggcaaaggccccgctgctcgaactcgacgtgcaaaacgaggagcacctggccagcttggccggccgggtgaccgaggcgatcggggcgggcaacaagctcgacggggtggtgcattcgattgggttcatgccgcagaccgggatgggcatcaacccgttcttcgacgcgccctacgcggatgtgtccaagggcatccacatctcggcgtattcgtatgcttcgatggccaaggcgctgctgccgatcatgaaccccggaggttccatcgtcggcatggacttcgacccgagccgggcgatgccggcctacaactggatgacggtcgccaagagcgcgttggagtcggtcaacaggttcgtggcgcgcgaggccggcaagtacggtgtgcgttcgaatctcgttgccgcaggccctatccggacgctggcgatgagtgcgatcgtcggcggtgcgctcggcgaggaggccggcgcccagatccagctgctcgaggagggctgggatcagcgcgctccgatcggctggaacatgaaggatgcgacgccggtcgccaagacggtgtgcgcgctgctgtctgactggctgccggcgaccacgggtgacatcatctacgccgacggcggcgcgcacacccaattgctcta'

In [37]:
myco_ref="atgacaggactgctggacggcaaacggattctggttagcggaatcatcaccgactcgtcgatcgcgtttcacatcgcacgggtagcccaggagcagggcgcccagctggtgctcaccgggttcgaccggctgcggctgattcagcgcatcaccgaccggctgccggcaaaggccccgctgctcgaactcgacgtgcaaaacgaggagcacctggccagcttggccggccgggtgaccgaggcgatcggggcgggcaacaagctcgacggggtggtgcattcgattgggttcatgccgcagaccgggatgggcatcaacccgttcttcgacgcgccctacgcggatgtgtccaagggcatccacatctcggcgtattcgtatgcttcgatggccaaggcgctgctgccgatcatgaaccccggaggttccatcgtcggcatggacttcgacccgagccgggcgatgccggcctacaactggatgacggtcgccaagagcgcgttggagtcggtcaacaggttcgtggcgcgcgaggccggcaagtacggtgtgcgttcgaatctcgttgccgcaggccctatccggacgctggcgatgagtgcgatcgtcggcggtgcgctcggcgaggaggccggcgcccagatccagctgctcgaggagggctgggatcagcgcgctccgatcggctggaacatgaaggatgcgacgccggtcgccaagacggtgtgcgcgctgctgtctgactggctgccggcgaccacgggtgacatcatctacgccgacggcggcgcgcacacccaattgctctag"

In [38]:
print(len(myco_ref),len(gene_sequence_data['Sequence'][0]))

810 809


In [39]:
start_index = myco_ref.find(gene_sequence_data['Sequence'][0])
start_index

0

In [40]:
from Bio.Data import CodonTable
import re
import numpy as np

def translate_sequence_with_gaps(dna_seq, table="Standard", to_stop=False, handle_stops='remove', ref_protein_length=None):
    codon_table = CodonTable.unambiguous_dna_by_name[table]
    standard_table = codon_table.forward_table
    stop_codons = codon_table.stop_codons

    dna_seq = dna_seq.upper()
    protein_seq = []
    frameshift_mutations = False

    seq_len = len(dna_seq)
    i = 0
    cumulative_gap_count = 0

    while i + 3 <= seq_len:
        codon = dna_seq[i:i+3]
        if '-' in codon:
            # Codon contains gap(s)
            cumulative_gap_count += codon.count('-')
            protein_seq.append('-')
        elif re.search(r'[^ATCG]', codon):
            # Codon contains ambiguous nucleotide(s)
            protein_seq.append('X')
        else:
            if codon in stop_codons:
                if to_stop:
                    break
                else:
                    if handle_stops == 'remove':
                        pass  # Ignore stop codon
                    elif handle_stops == 'replace':
                        protein_seq.append('X')  # Replace stop codon with X
                    else:
                        protein_seq.append('*')  # Keep stop codon
            else:
                amino_acid = standard_table.get(codon, 'X')
                protein_seq.append(amino_acid)
        i += 3

    # Handle remaining nucleotides at the end
    if i < seq_len:
        remaining = dna_seq[i:]
        if '-' in remaining or re.search(r'[^ATCG]', remaining):
            protein_seq.append('-')
        else:
            # Remaining nucleotides less than a codon length (ignore instead of flagging)
            pass

    protein_str = ''.join(protein_seq)

    # **Fix: Prevent unnecessary frameshift flagging**
    if ref_protein_length is not None:
        translated_length = len(protein_seq)

        # Allow small mismatches (e.g., ±5% tolerance)
        length_difference = abs(translated_length - ref_protein_length)
        length_tolerance = max(1, int(ref_protein_length * 0.05))

        if length_difference > length_tolerance:
            frameshift_mutations = True  # Large length mismatch
        else:
            frameshift_mutations = False  # Small variations are fine

    # **Fix: Only flag internal stop codons, not the last one**
    if '*' in protein_str[:-1]:  # Ignore the last stop codon
        frameshift_mutations = True

    return protein_str, frameshift_mutations


In [67]:
def align_and_handle_deletions(translated_seq, ref_seq):
    """Handle deletions and ensure proper alignment with placeholders for gaps."""
    aligned_seq = []
    ref_index = 0
    trans_index = 0

    while ref_index < len(ref_seq) and trans_index < len(translated_seq):
        if translated_seq[trans_index] == ref_seq[ref_index]:
            # If they match, add the amino acid from the translated sequence
            aligned_seq.append(translated_seq[trans_index])
            trans_index += 1
        else:
            if translated_seq[trans_index] == '-':
                # If there is a gap in the translated sequence, mark it as a gap
                aligned_seq.append('-')
                trans_index += 1
            elif ref_seq[ref_index] == '-':
                # If the reference has a gap, skip the reference gap
                ref_index += 1
            else:
                # If it's a substitution (mismatch), append the amino acid from the translated sequence
                aligned_seq.append(translated_seq[trans_index])
                trans_index += 1

        # Move the reference index forward regardless
        ref_index += 1

    # **Fix: Trim excess gaps at the end**
    # aligned_seq_str = ''.join(aligned_seq).rstrip('-')
    aligned_seq_str = ''.join(aligned_seq)

    return aligned_seq_str


In [84]:
def convert_to_onehot_with_reference(aa_seq, ref_aa):
    # return np.array([1 if aa == ref else 0 for aa, ref in zip(aa_seq, ref_aa)])
    return np.array([0 if aa == ref else 1 for aa, ref in zip(aa_seq, ref_aa)])

def encode_sequence(sequence, reference_length, h37rv_aa_str):
    encoded = convert_to_onehot_with_reference(str(sequence), str(h37rv_aa_str))
    return encoded

In [85]:
all_translations = []
all_labels = []
problematic_indices = set()
frameshift_mutations_list = []

for i in range(len(gene_sequence_data)):
    problematic = False
    if orientation == 'plus':
        h37rv_aa_str = Seq(h37rv_sequence_str).translate()
        reference_length = len(h37rv_aa_str)
        translation, frameshift = translate_sequence_with_gaps(gene_sequence_data["Sequence"][i], to_stop=False, ref_protein_length=reference_length)
    else:
        h37rv_aa_str = Seq(h37rv_sequence_str).reverse_complement().translate()
        reference_length = len(h37rv_aa_str)
        template_dna = Seq(gene_sequence_data["Sequence"][i]).reverse_complement()
        translation, frameshift = translate_sequence_with_gaps(str(template_dna), to_stop=False, ref_protein_length=reference_length)
    
    aligned_translation = align_and_handle_deletions(translation, h37rv_aa_str)


    if i == 1:
        print("translation before aligning", translation)
        print("translation after aligning", aligned_translation)
        print(f"Translated length: {len(translation)}, Reference length: {reference_length}")
        print(f"Final aligned sequence: {aligned_translation}")

    if frameshift:
        problematic = True
        problematic_indices.add(i)
        frameshift_mutations_list.append(1)
        aligned_translation = '0' * reference_length
    else:
        frameshift_mutations_list.append(0)

    # print(f"Row {row_index}: frameshift={frameshift}, aligned_translation={aligned_translation[:50]}...")
    all_translations.append(aligned_translation)
    all_labels.append(gene_sequence_data["Phenotype"][i])

# **Add new columns to the DataFrame**
gene_sequence_data["Protein_Sequence"] = all_translations
gene_sequence_data["Frameshift_Mutation"] = frameshift_mutations_list

/work/pi_annagreen_umass_edu/mahbuba/protein/lib/python3.10/site-packages/Bio/Seq.py:2804: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


translation before aligning MTGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRLIQRITDRLPAKAPLLELDVQNEEHLASLAGRVTEAIGAGNKLDGVVHSIGFMPQTGMGINPFFDAPYADVSKGIHISAYSYASMAKALLPIMNPGGSIVGMDFDPSRAMPAYNWMTVAKSALESVNRFVAREAGKYGVRSNLVAAGPIRTLAMSAIVGGALGEEAGAQIQLLEEGWDQRAPIGWNMKDATPVAKTVCALLSDWLPATTGDIIYADGGAHTQLL
translation after aligning MTGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRLIQRITDRLPAKAPLLELDVQNEEHLASLAGRVTEAIGAGNKLDGVVHSIGFMPQTGMGINPFFDAPYADVSKGIHISAYSYASMAKALLPIMNPGGSIVGMDFDPSRAMPAYNWMTVAKSALESVNRFVAREAGKYGVRSNLVAAGPIRTLAMSAIVGGALGEEAGAQIQLLEEGWDQRAPIGWNMKDATPVAKTVCALLSDWLPATTGDIIYADGGAHTQLL
Translated length: 269, Reference length: 269
Final aligned sequence: MTGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRLIQRITDRLPAKAPLLELDVQNEEHLASLAGRVTEAIGAGNKLDGVVHSIGFMPQTGMGINPFFDAPYADVSKGIHISAYSYASMAKALLPIMNPGGSIVGMDFDPSRAMPAYNWMTVAKSALESVNRFVAREAGKYGVRSNLVAAGPIRTLAMSAIVGGALGEEAGAQIQLLEEGWDQRAPIGWNMKDATPVAKTVCALLSDWLPATTGDIIYADGGAHTQLL


In [86]:
translation==aligned_translation

True

In [87]:
len(translation), len(aligned_translation)

(269, 269)

In [88]:
h37rv_aa_str

Seq('MTGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRLIQRITDRL...QLL')

### add protein sequence to the gene csv file and save

In [89]:
output_file_name=gene_name+"_sequence_data.csv"
gene_sequence_data.to_csv("../data/sequence_data_csv/"+output_file_name, index=False)

In [90]:
## data exploration step
unique_elements, counts = np.unique(frameshift_mutations_list, return_counts=True)
value_counts_dict = dict(zip(unique_elements, counts))
print(value_counts_dict)

problem_percentage = (len(problematic_indices) / len(gene_sequence_data)) * 100
print(f"Problematic percentage: {problem_percentage}%")

print("Keeping all sequences.")
valid_indices = range(len(gene_sequence_data))

valid_translations = [(filenames[i], all_translations[i][:reference_length]) for i in valid_indices]
valid_labels = [all_labels[i] for i in valid_indices]

valid_lengths = [len(seq[1]) for seq in valid_translations]
length_mismatches = [i for i, length in enumerate(valid_lengths) if length != reference_length]
total_sequences = len(valid_translations)
num_mismatches = len(length_mismatches)
print(f"Reference length expected: {reference_length}")
print(f"Sample of sequence lengths after truncation: {[len(seq[1]) for seq in valid_translations[:10]]}")

if num_mismatches == 0:
    print("All translations have the correct length.")
else:
    mismatch_percentage = (num_mismatches / total_sequences) * 100
    print(f"{num_mismatches} sequences have varying lengths out of {total_sequences} sequences.")
    print(f"Percentage of sequences with varying lengths: {mismatch_percentage:.2f}%")
    # print(f"Indices of sequences with varying lengths: {length_mismatches}")
    # print(f"Lengths of the mismatched sequences: {[valid_lengths[i] for i in length_mismatches]}")

print(f"Total sequences after filtering: {len(valid_translations)}")
print(f"Sample of sequence lengths after filtering: {[len(seq[1]) for seq in valid_translations[:10]]}")

{0: 29446}
Problematic percentage: 0.0%
Keeping all sequences.
Reference length expected: 269
Sample of sequence lengths after truncation: [269, 269, 269, 269, 269, 269, 269, 269, 269, 269]
1 sequences have varying lengths out of 29446 sequences.
Percentage of sequences with varying lengths: 0.00%
Total sequences after filtering: 29446
Sample of sequence lengths after filtering: [269, 269, 269, 269, 269, 269, 269, 269, 269, 269]


In [91]:
gene_sequence_data

,Filename,Sequence,Phenotype,seq_len,Protein_Sequence,Frameshift_Mutation
0,00R0025,atgacaggactgctggacggcaaacggattctggttagcggaatca...,R,809,MTGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRL...,0
1,00R0086,atgacaggactgctggacggcaaacggattctggttagcggaatca...,R,809,MTGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRL...,0
2,00R0178,atgacaggactgctggacggcaaacggattctggttagcggaatca...,R,809,MTGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRL...,0
3,00R0223,atgacaggactgctggacggcaaacggattctggttagcggaatca...,S,809,MTGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRL...,0
4,00R0312,atgacaggactgctggacggcaaacggattctggttagcggaatca...,R,809,MTGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRL...,0
...,...,...,...,...,...,...
29441,ERR4813816,atgacaggactgctggacggcaaacggattctggttagcggaatca...,S,809,MTGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRL...,0
29442,ERR4813817,atgacaggactgctggacggcaaacggattctggttagcggaatca...,S,809,MTGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRL...,0
29443,ERR4813863,atgacaggactgctggacggcaaacggattctggttagcggaatca...,R,809,MTGLLDGKRILVSGIITDSSVAFHIARVAQEQGAQLVLTGFDRLRL...,0
29444,ERR4813864,atgacaggactgctggacggcaaacggattctggttagcggaatca...,R,809,MTGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRL...,0


In [92]:
myco_ref="MTGLLDGKRILVSGIITDSSIAFHIARVAQEQGAQLVLTGFDRLRLIQRITDRLPAKAPLLELDVQNEEHLASLAGRVTEAIGAGNKLDGVVHSIGFMPQTGMGINPFFDAPYADVSKGIHISAYSYASMAKALLPIMNPGGSIVGMDFDPSRAMPAYNWMTVAKSALESVNRFVAREAGKYGVRSNLVAAGPIRTLAMSAIVGGALGEEAGAQIQLLEEGWDQRAPIGWNMKDATPVAKTVCALLSDWLPATTGDIIYADGGAHTQLL"

In [93]:
len(myco_ref)

269

In [94]:
start_index = myco_ref.find(gene_sequence_data['Protein_Sequence'][0])
start_index

0

In [95]:
print(len(aligned_translation),len(gene_sequence_data['Protein_Sequence'][0]))

269 269


In [96]:
myco_ref[61:]

'ELDVQNEEHLASLAGRVTEAIGAGNKLDGVVHSIGFMPQTGMGINPFFDAPYADVSKGIHISAYSYASMAKALLPIMNPGGSIVGMDFDPSRAMPAYNWMTVAKSALESVNRFVAREAGKYGVRSNLVAAGPIRTLAMSAIVGGALGEEAGAQIQLLEEGWDQRAPIGWNMKDATPVAKTVCALLSDWLPATTGDIIYADGGAHTQLL'

### write aa fasta file with filename, aa seq, phenotype and frameshift flag

In [97]:
# Define output file path
protein_name = f'../data/aa_fasta/all_sequences_{gene_name}_aa.fasta'

# Ensure the DataFrame has the correct columns
if {"Filename", "Protein_Sequence", "Phenotype", "Frameshift_Mutation"}.issubset(gene_sequence_data.columns):
    write_fasta_with_metadata_from_df(gene_sequence_data, protein_name, reference_length)
    print(f"FASTA file saved at {protein_name}")
else:
    print("Error: The DataFrame is missing required columns.")


FASTA file saved at ../data/aa_fasta/all_sequences_inhA_aa.fasta


In [99]:

filtered_translations = [(fn, seq) for (fn, seq) in valid_translations if len(seq) == reference_length]

filtered_labels = [valid_labels[i] for i, (_, seq) in enumerate(valid_translations) if len(seq) == reference_length]

filtered_flags = [frameshift_mutations_list[i] for i, (_, seq) in enumerate(valid_translations) if len(seq) == reference_length]


print(f"Reference length expected: {reference_length}")
print(f"Total sequences before filtering: {len(all_translations)}")
print(f"Total sequences after filtering: {len(filtered_translations)}")
print(f"Dropped {len(all_translations) - len(filtered_translations)} sequences.")



Reference length expected: 269
Total sequences before filtering: 29446
Total sequences after filtering: 29445
Dropped 1 sequences.


In [100]:

# --------- Step 2: Encode Sequences ---------

num_cores = joblib.cpu_count()

encoded_features = Parallel(n_jobs=num_cores)(
    delayed(encode_sequence)(seq, reference_length, h37rv_aa_str)
    for _, seq in filtered_translations
)

# Sanity check
lengths = [len(f) for f in encoded_features]
assert all(l == reference_length for l in lengths), "Inconsistent feature lengths after encoding!"
print(f"Encoding complete. Unique encoded lengths: {set(lengths)}")


Encoding complete. Unique encoded lengths: {269}


### save feature matrix and feature labels

In [101]:

# --------- Step 3: Assemble Feature Matrix ---------

mutation_flags = np.array(filtered_flags).reshape(-1, 1)
encoded_matrix = np.array(encoded_features)
feature_matrix = np.column_stack((mutation_flags, encoded_matrix))

print(f"Final feature matrix shape: {feature_matrix.shape}")

# --------- Step 4: Save ---------

np.save(f'../data/feature_matrix_labels/{gene_name}_feature_matrix.npy', feature_matrix)
np.save(f'../data/feature_matrix_labels/{gene_name}_labels.npy', filtered_labels)

Final feature matrix shape: (29445, 270)


# Data loading for Model Training

In [102]:
X, y = load_feature_matrix_and_labels(gene_name)

/work/pi_annagreen_umass_edu/mahbuba/phenotype_prediction/fused_lasso/latest_codes/feature_matrix_labels/inhA_feature_matrix.npy
Loading feature matrix and labels for inhA from disk.


In [103]:
X.shape

(29446, 270)

# full sequence with ridge/lasso

In [104]:
# try with full sequence first
y_encoded = encode_labels(y)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42)
print(f"Train data shape before deduplication: {np.array(X_train).shape}")
full_train_shape = np.array(X_train).shape


lasso = LassoCV(max_iter=10000, cv=5, random_state=42, alphas=[0.001, 0.01, 0.1, 1, 10, 100])
# lasso = LassoCV()
lasso.fit(X_train,y_train)

regular_lasso_R2 = lasso.score(X_test, y_test)
print("Lasso Score", regular_lasso_R2)
# logging.info(f"Lasso Score: {regular_lasso_R2}")
regular_lasso_auc =sklearn.metrics.roc_auc_score(y_test, lasso.predict(X_test))
print("Lasso AUC", regular_lasso_auc)
# logging.info(f"Lasso AUC: {regular_lasso_auc}")
regular_lasso_mse = mean_squared_error(y_test, lasso.predict(X_test))
print(f"regular lasso mse:{regular_lasso_mse}")


ridge = RidgeCV(alphas=[0.001, 0.01, 0.1, 1, 10]).fit(X_train, y_train)
full_ridge_score = ridge.score(X_test, y_test)
full_ridge_auc = sklearn.metrics.roc_auc_score(y_test, ridge.predict(X_test))
full_ridge_mse = mean_squared_error(y_test, ridge.predict(X_test))
print("ridge score", full_ridge_score)
print("ridge mse", full_ridge_mse)
print("ridge auc", full_ridge_auc)

logistic_model = LogisticRegressionCV(cv=3, scoring="roc_auc", max_iter=5000,Cs=[1e-4, 1e-3, 1e-2, 0.1, 1, 10, 100], class_weight="balanced").fit(X_train, y_train)
log_score = logistic_model.score(X_test, y_test)
log_auc = sklearn.metrics.roc_auc_score(y_test, logistic_model.predict(X_test))
log_mse = mean_squared_error(y_test, logistic_model.predict(X_test))

print("log score", log_score)
print("log mse", log_mse)
print("log auc", log_auc)


Train data shape before deduplication: (23556, 270)
Lasso Score 0.02142878835336237
Lasso AUC 0.5246947759125246
regular lasso mse:0.229857202056246
ridge score 0.027827125957629484
ridge mse 0.22835429254692846
ridge auc 0.5306929849847506
log score 0.5304909932678834
log mse 0.3606112054329372
log auc 0.5239693360375189


In [20]:
# Define Paths
output_feature_dir = "/work/pi_annagreen_umass_edu/mahbuba/phenotype_prediction/data/spectra/"+ drug_code
split_dir = "/work/pi_annagreen_umass_edu/mahbuba/SPECTRA/SPECTRA_paper/SPECTRA_MSD/TB_Covid_GFP/data/"+ drug_code+"_binary_mutational_split/"
gene_aa= "all_sequences_"+gene_name+"_aa.fasta"
alignment_path = "/work/pi_annagreen_umass_edu/mahbuba/phenotype_prediction/fused_lasso/latest_codes/"+gene_aa

lambda_values = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]
splits = [0, 1, 2]

# run on unique de-duplicated data

In [105]:
variants_df
# Extracting information using regex
variants_df[['aa_ref', 'aa_pos', 'aa_change']] = variants_df['mutation'].str.extract(r'p\.([A-Za-z]+)(\d+)([A-Za-z]+)')


In [106]:
filtered_df = variants_df[variants_df['gene'].str.contains(gene_name, case=False, na=False)]

# Extract the amino acid position directly from the 'aa_pos' column
filtered_df.dropna(subset=['aa_pos'], inplace=True)

/tmp/ipykernel_2488110/177593172.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df.dropna(subset=['aa_pos'], inplace=True)


In [107]:
gene

'inhA'

In [108]:
# Filter rows where the 'confidence' column contains "Assoc w R"
filtered_df = filtered_df[filtered_df['confidence'].str.contains("Assoc w R", case=False, na=False)]

# Extract the total unique amino acid positions after filtering
total_actual_positives = len(np.unique(filtered_df['aa_pos']))

print(f"Total actual positives (Assoc w R): {total_actual_positives}")

Total actual positives (Assoc w R): 2


In [109]:
# Preprocess data-deduplication
X_unique, original_indices = drop_identical_columns(X)
X_unique_rows, row_selection_indices = drop_identical_sequences(X_unique)
ys = y[row_selection_indices]
phenotypes, [X_unique_rows_filtered] = filter_nan_labels(ys, X_unique_rows)
y_encoded = encode_labels(phenotypes)


X_train, X_test, y_train, y_test = train_test_split(X_unique_rows_filtered, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42)
X_train = np.array(X_train)
X_test=np.array(X_test)
y_train = np.array(y_train)
y_test = np.array(y_test)
print("data shape after deduplication", np.array(X_unique_rows_filtered).shape)

data shape after deduplication (103, 65)


In [110]:
original_indices = list(range(X_train.shape[1]))
print(original_indices[:10])  # Print first 10 indices for verification

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [111]:
print(X_train.shape)

(82, 65)


In [112]:
len(X_unique_rows_filtered)

103

In [114]:
# Train models
ridge_model, ridge_score, ridge_auc, ridge_mse = train_ridge_model(X_train, y_train, X_test, y_test)
print("ridge performance", ridge_auc)

initial_coef= ridge_model.coef_

ridge performance 0.6636363636363637
